# BP3 — Notebook 3 (re-run to generate the missing lookup Parquet)

**This is a throwaway execution wrapper, not a new canonical file.** The project's canonical source is `notebooks/bp3_network_graph_intelligence/03_statistical_validation_deployment_SINGLE_CELL.py` (unedited) -- this notebook just pastes that file's exact content into one cell, per that file's own header instruction. Don't commit this .ipynb.

This generates `models/bp3_network_graph_intelligence/bp3_notebook3_near_train_flagged_lookup_li_medium.parquet` (DEF-002) -- this BP has no checkpoint/resume mechanism (unlike BP2's), so this is a full run from scratch: reads the real LI-Medium CSV (31.25M rows), builds the account graph, computes the 2-hop proximity signal, and this time reaches the lookup-table save step a prior run (2026-10-01, which produced `bp3_notebook3_champion_rule_li_medium.json` but not the lookup Parquet) did not reach. No code was changed from the committed .py.

**Watch for:**
- Progress prints at each `thermal_checkpoint(...)` label (post composite-key encoding / post degree+PageRank / post 2-hop proximity BFS / post bootstrap CI) — these confirm forward progress even though there's no resume.
- `Real near_train_flagged lookup (N real accounts) saved to: ...parquet` near the end -- this is the real completion signal for the missing artifact specifically.
- Any `RuntimeError` — stop and send me the full traceback rather than re-running blind.

**After it finishes, confirm these files exist / were updated:**
- `models/bp3_network_graph_intelligence/bp3_notebook3_near_train_flagged_lookup_li_medium.parquet (new file)`


In [ ]:
# ============================================================================
# BP3 NOTEBOOK 3 -- Statistical Validation & Deployment (SINGLE CELL -- paste this whole file into one Jupyter cell)
# ============================================================================
#
# Purpose (per the locked master-execution-plan, Section 4 + Section 3's per-BP validation-tier table):
# re-run BP3's real structural-signal pipeline -- IDENTICAL graph-construction/degree/PageRank/proximity
# logic to Notebook 2 -- against BP3's locked mandatory realism-validation tier: LI-Medium (31,251,484 real
# transaction rows, confirmed this session by a real `wc -l` on the user's own device). Re-confirms Notebook
# 2's real HI-Small Stage A champion signal at real scale, adds a real bootstrap 95% confidence interval on
# the champion's network-lift ratio, the two-gate verdict, and the deployable FastAPI scoring service.
#
# **No trained ML model in this BP (unchanged from Notebook 2's own disclosure):** every candidate is a
# directly-interpretable structural signal. SHAP/LIME are N/A here -- not attempted, not fabricated.
#
# **Real engineering change from the "obvious" approach, disclosed honestly -- NEW this notebook, not
# carried from Notebook 2 (HI-Small was small enough that this didn't matter there):** Notebook 1's own real
# pre-build diagnostic (Lesson #21) already found that `networkx`'s pure-Python graph algorithms show real,
# serious scaling problems even on HI-Small, the SMALLEST real variant (Louvain did not finish in 120s).
# LI-Medium is far larger. Rather than build a full `networkx.DiGraph` here and discover the same class of
# problem again, this notebook was engineered and REAL-TESTED before being written:
#   1. Degree: vectorized pandas groupby (identical method to Notebook 2 -- already proven real-cheap).
#   2. PageRank: a hand-rolled sparse-matrix power-iteration (scipy.sparse + numpy, fully vectorized, no
#      `networkx.DiGraph` object ever constructed for the full graph) -- verified CORRECT by Claude against
#      `networkx.pagerank()`'s own reference output on a real local synthetic graph (400 nodes, 2000 edges):
#      max absolute difference 1.45e-06, identical top-5 ranking. This exact verification is reproduced
#      below as this notebook's own real, live correctness self-test (Section 3) -- not just asserted from
#      memory of a prior session.
#   3. The 2-hop proximity signal: a memory-efficient vectorized frontier-expansion BFS (two directed
#      pandas merges per hop -- outgoing and incoming -- against the real edge table), never a `networkx`
#      graph traversal over the full graph. A REAL bug was caught and fixed before shipping this: an
#      earlier version built a full undirected, deduplicated edge table (`pd.concat` of both directions +
#      `drop_duplicates()`) before expanding the frontier -- this DOUBLED real edge storage and the
#      dedup step alone was enough to OOM-kill a synthetic test at LI-Medium's real scale (31,251,484
#      synthetic edges, 2,000,000 synthetic nodes) in a real, measured 7.8GB-RAM test container, confirmed
#      via a real `exit code 137`. Fixed by never materializing the doubled/deduped table -- each hop does
#      two small directed merges (frontier-as-src, frontier-as-dst) instead. Full write-up: Lesson #26 in
#      `LESSONS_LEARNED_APPLIED.md`.
#   4. Honest disclosure on WHERE this was tested: NOT a live run against the real LI-Medium file on the
#      user's own device. The on-device sandbox this platform's own Claude session uses for file operations
#      has only ~4GB total RAM -- materially less than this platform's real 8-core/16-thread/32GB target
#      machine -- so a timing/memory number measured there would not be representative (and risks a
#      misleading OOM that says nothing about the real target machine). Instead, the scaling test above ran
#      in Claude's own cloud test environment (~7.8GB RAM) against a SYNTHETIC graph matching LI-Medium's
#      real confirmed node/edge-count order of magnitude: real measured wall-clock was ~36s (distinct-edge
#      aggregation) + ~12s (sparse PageRank, matrix build + power iteration) + ~8s (2-hop frontier BFS for
#      20,000 real seed-equivalent nodes), peak RSS ~4.2GB -- well inside this platform's real 32GB ceiling.
#      This notebook's OWN `timer`/`timed` instrumentation prints the real, authoritative number for the
#      user's actual machine when this cell actually runs -- the figures above are Claude's own pre-build
#      feasibility evidence, not a substitute for that real run, per the same honesty discipline as Lesson
#      #21's own Louvain finding.
#   5. `networkx` is still used, deliberately, for the small, bounded, 2,000-node-capped illustrative
#      example ego-networks (Notebook 2 Section 9's pattern, reused here) -- safe at that scale, and the
#      one place the locked graph-feasibility policy (Notebook 1 Section 5) actually calls for it.
#
# **Bootstrap CI methodology (Section 9.3 technique #5 -- closed-form multinomial resampling, never a
# brute-force per-resample graph rebuild):** the champion signal's real lift ratio is a function of a real
# 2x2 contingency table (flagged x exposed, among TEST nodes). A real 95% CI is built by drawing 2,000 real
# multinomial resamples directly from that table's own real cell probabilities (one batched
# `rng.multinomial` call, Section 9.3 technique #6 -- never a per-resample Python loop), recomputing the
# real lift ratio for every resample, and reporting the real 2.5th/97.5th percentile.
#
# Zero-fabrication: every number below is computed live, on whichever real CSV DATASET_VARIANT points at
# (locked default: LI-Medium). RANDOM_SEED=42 throughout.
# ============================================================================

import sys
import gc
import platform
from pathlib import Path

if platform.system() == "Windows":
    import ctypes
    _ES_CONTINUOUS = 0x80000000
    _ES_SYSTEM_REQUIRED = 0x00000001
    _sleep_block_result = ctypes.windll.kernel32.SetThreadExecutionState(_ES_CONTINUOUS | _ES_SYSTEM_REQUIRED)
    if _sleep_block_result == 0:
        print("WARNING: SetThreadExecutionState call failed (returned 0) -- sleep prevention may "
              "not be active. Rely on the OS power-setting change as the primary safeguard.")
    else:
        print("Sleep prevention ACTIVE for this kernel process -- Windows will not suspend the "
              "system while this Jupyter kernel is running (display may still sleep separately; "
              "that does not affect the running computation).")
else:
    print(f"Sleep-prevention block skipped -- not running on Windows (platform.system()="
          f"{platform.system()!r}).")

def _locate_project_root():
    cur = Path.cwd()
    for _ in range(8):
        if (cur / "PROJECT_STRUCTURE_LOCKED.md").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    known = Path.home() / "Documents" / "IBM_AML_RiskIQ_Enterprise_Suite"
    if (known / "PROJECT_STRUCTURE_LOCKED.md").exists():
        return known
    raise FileNotFoundError(
        "Could not locate the project root. Checked: an upward walk from this notebook's "
        f"working directory ({Path.cwd()}), and {known}. Either save this notebook inside "
        "the project folder, or edit the 'known' path above to your real project location."
    )

_here = _locate_project_root()
sys.path.insert(0, str(_here / "src" / "aml_riskiq"))
print(f"Bootstrapped from: {_here}")

from utils.project_root import find_suite_root, raw_data_dir
from utils.performance_setup import (
    configure_performance, thermal_checkpoint, load_csv_cached, timer,
    check_ram_headroom, pin_cpu_affinity, assert_ram_safe,
)

perf_config = configure_performance()
print("WARP configured:", perf_config)
_affinity_pinned = pin_cpu_affinity()
print("CPU affinity pin (best-effort, real):", _affinity_pinned)
print("RAM headroom baseline (real):", check_ram_headroom())

import psutil
def _mem_gb():
    return psutil.Process().memory_info().rss / (1024 ** 3)

def _print_resource(label):
    print(f"[RESOURCE] {label} -- process RSS: {_mem_gb():.2f} GB | {check_ram_headroom()}")

import json
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from scipy import sparse

try:
    import networkx as nx
    print(f"networkx {nx.__version__} -- available (used only for the small, capped illustrative "
          f"ego-networks, Section 7 -- never the full graph, per the engineering decision above).")
except ImportError as e:
    raise ImportError(f"networkx is required for this notebook's bounded illustrative ego-networks "
                       f"(Section 7) and is not installed ({e}). ACTION NEEDED: pip install networkx.")

RANDOM_SEED = 42
# HOW TO RUN THIS NOTEBOOK: LI-Medium is this BP's locked mandatory realism-validation tier (Notebook 1
# Section 5 / master-plan Section 2.1). Change ONLY this one line to re-point at a different real variant.
DATASET_VARIANT = "LI-Medium"
TRANS_FILE = f"{DATASET_VARIANT}_Trans.csv"

PROJECT_ROOT = find_suite_root()
RAW = raw_data_dir()
CONFIG_PATH = PROJECT_ROOT / "configs" / "bp3_network_graph_intelligence.yaml"
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp3_network_graph_intelligence"
MODELS_DIR = PROJECT_ROOT / "models" / "bp3_network_graph_intelligence"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root  : {PROJECT_ROOT}")
print(f"Reports dir   : {REPORTS_DIR}")
print(f"Models dir    : {MODELS_DIR}")
print(f"Dataset variant: {DATASET_VARIANT}")

def _read_simple_yaml(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        k, v = line.split(":", 1)
        out[k.strip()] = v.strip().strip('"')
    return out

bp3_config = _read_simple_yaml(CONFIG_PATH)
if bp3_config.get("status") == "scaffolded - not yet built":
    raise RuntimeError(
        "configs/bp3_network_graph_intelligence.yaml still carries the pre-run placeholder. Run BP3 "
        "Notebook 1 (and ideally Notebook 2, for a real HI-Small Stage A prior) first."
    )
PRIMARY_METRIC = bp3_config.get("primary_metric", "network_lift_ratio")
EGO_HOPS = 2
EGO_NODE_CAP = 2000

# Real HI-Small Stage A prior from Notebook 2 (if present) -- re-confirmed here at real LI-Medium scale,
# never assumed to still hold without this notebook's own real recomputation.
nb2_summary_path = REPORTS_DIR / "bp3_notebook2_stage_a_summary_hi_small.json"
nb2_champion_prior = None
if nb2_summary_path.exists():
    with open(nb2_summary_path, "r", encoding="utf-8") as f:
        nb2_champion_prior = json.load(f).get("champion_signal")
    print(f"\nReal Notebook 2 (HI-Small) Stage A champion prior: {nb2_champion_prior!r} -- re-confirmed "
          f"(or not) at real {DATASET_VARIANT} scale below, never assumed to carry over unchanged.")
else:
    print(f"\nNo Notebook 2 Stage A summary found at {nb2_summary_path} -- proceeding without a real prior "
          f"(not a blocking requirement, but run Notebook 2 first for a real HI-Small comparison point).")

# ============================================================================
# 0. Real correctness self-test -- hand-rolled sparse PageRank vs. networkx's own reference, on a small
#    local synthetic graph. This is NOT this notebook's real result (that comes from Section 3, against
#    the real LI-Medium graph) -- it is a live, reproducible proof that the vectorized implementation used
#    below is correct, run fresh every time this cell runs, never just asserted from a prior session.
# ============================================================================
def sparse_pagerank(n_nodes, src, dst, weight=None, alpha=0.85, max_iter=100, tol=1e-10):
    """Real, vectorized sparse-matrix power-iteration PageRank -- scipy.sparse + numpy only, never a
    `networkx.DiGraph` object. Dangling nodes (real out-degree 0) redistribute their rank uniformly across
    ALL nodes, matching networkx's own convention (verified below). `src`/`dst` are real compact int node
    ids; `weight` defaults to uniform (1 per edge) if not given."""
    weight = np.ones(len(src), dtype="float64") if weight is None else np.asarray(weight, dtype="float64")
    out_deg = np.zeros(n_nodes, dtype="float64")
    np.add.at(out_deg, src, weight)
    safe_out_deg = np.where(out_deg == 0, 1.0, out_deg)  # avoid div-by-zero; those rows are never used (w=0 contribution masked below is not needed since dangling rows have no real outgoing edges)
    w = weight / safe_out_deg[src]
    M = sparse.csr_matrix((w, (src, dst)), shape=(n_nodes, n_nodes))
    dangling = (out_deg == 0).astype("float64")
    rank = np.full(n_nodes, 1.0 / n_nodes, dtype="float64")
    for _ in range(max_iter):
        dangling_mass = float((rank * dangling).sum())
        new_rank = alpha * (rank @ M) + alpha * dangling_mass / n_nodes + (1 - alpha) / n_nodes
        new_rank = np.asarray(new_rank).ravel()
        if np.abs(new_rank - rank).sum() < tol:
            rank = new_rank
            break
        rank = new_rank
    return rank

with timer("real self-test: sparse PageRank vs. networkx reference (small synthetic graph)"):
    _rng = np.random.default_rng(3)
    _n = 400
    _edges = list({(int(a), int(b)) for a, b in _rng.integers(0, _n, size=(2000, 2)) if a != b})
    _src = np.array([e[0] for e in _edges])
    _dst = np.array([e[1] for e in _edges])
    _Gref = nx.DiGraph()
    _Gref.add_nodes_from(range(_n))
    _Gref.add_edges_from(_edges)
    _pr_ref = nx.pagerank(_Gref, alpha=0.85)
    _pr_mine = sparse_pagerank(_n, _src, _dst, alpha=0.85)
    _max_abs_diff = float(max(abs(_pr_ref[i] - _pr_mine[i]) for i in range(_n)))
if _max_abs_diff > 1e-4:
    raise AssertionError(
        f"Real PageRank self-test FAILED: hand-rolled sparse implementation diverges from networkx's own "
        f"reference by {_max_abs_diff:.2e} (threshold 1e-4) on this run's real synthetic check graph. "
        f"STOPPING -- do not trust this notebook's PageRank numbers below until this is fixed."
    )
print(f"Real PageRank self-test PASSED: max abs diff vs. networkx reference = {_max_abs_diff:.2e} "
      f"(threshold 1e-4) on a {_n}-node / {len(_edges)}-edge real local synthetic graph.")
del _rng, _edges, _src, _dst, _Gref, _pr_ref, _pr_mine
gc.collect()

# ============================================================================
# 1. Load real Trans.csv at this BP's mandatory realism-validation tier (WARP: Parquet-cached).
# ============================================================================
TRANS_DTYPES = {
    "From Bank": "string", "Account": "string", "To Bank": "string", "Account.1": "string",
    "Amount Received": "float64", "Receiving Currency": "category",
    "Amount Paid": "float64", "Payment Currency": "category",
    "Payment Format": "category", "Is Laundering": "int8",
}
assert_ram_safe(min_available_gb=3.0, label=f"before loading {TRANS_FILE}")
with timer(f"load {TRANS_FILE} (WARP: cached to Parquet)"):
    trans = load_csv_cached(RAW / TRANS_FILE, parse_dates=["Timestamp"], dtype=TRANS_DTYPES)
print(f"Real {TRANS_FILE} shape: {trans.shape}")
_print_resource("post Trans.csv load")

# ============================================================================
# 2. Real composite-key graph encoding -- compact int node ids, vectorized (identical method to
#    Notebook 2; NO `networkx.DiGraph` object built for the full graph -- see header).
# ============================================================================
assert_ram_safe(min_available_gb=4.0, label="before composite-key encoding + distinct-edge aggregation")
with timer("real composite-key encoding + distinct-edge aggregation (vectorized)"):
    from_key = trans["From Bank"].astype(str) + "|" + trans["Account"].astype(str)
    to_key = trans["To Bank"].astype(str) + "|" + trans["Account.1"].astype(str)
    n_edges_raw = len(trans)

    all_keys = pd.concat([from_key, to_key], ignore_index=True)
    codes, uniques = pd.factorize(all_keys, sort=False)
    from_code = codes[:n_edges_raw]
    to_code = codes[n_edges_raw:]

    node_df = pd.DataFrame({"node_id": np.arange(len(uniques), dtype="int64"), "node_key": uniques})
    n_nodes = len(node_df)

    edge_df = (
        pd.DataFrame({"src": from_code, "dst": to_code})
        .groupby(["src", "dst"]).size().reset_index(name="weight")
    )
    n_distinct_edges = len(edge_df)

print(f"Real graph (compact-int encoded, no networkx object): {n_nodes:,} nodes, {n_distinct_edges:,} "
      f"distinct directed edges ({n_edges_raw:,} real transaction-edges collapsed by (src,dst) weight).")
thermal_checkpoint(label="post composite-key encoding")
_print_resource("post composite-key encoding")

# ============================================================================
# 3. Real degree (vectorized pandas) + real PageRank (hand-rolled sparse power-iteration, Section 0's
#    self-tested implementation -- THIS is the notebook's own real result, on the real graph).
# ============================================================================
with timer("real degree computation (vectorized)"):
    out_deg = pd.Series(to_code).groupby(from_code).nunique()
    in_deg = pd.Series(from_code).groupby(to_code).nunique()
    node_df["out_degree"] = node_df["node_id"].map(out_deg).fillna(0).astype("int32")
    node_df["in_degree"] = node_df["node_id"].map(in_deg).fillna(0).astype("int32")
    node_df["total_degree"] = node_df["out_degree"] + node_df["in_degree"]

assert_ram_safe(min_available_gb=4.0, label="before real sparse PageRank (full graph)")
with timer("real sparse PageRank (full graph, vectorized, self-tested implementation)"):
    pagerank_arr = sparse_pagerank(
        n_nodes, edge_df["src"].to_numpy(), edge_df["dst"].to_numpy(), weight=edge_df["weight"].to_numpy()
    )
node_df["pagerank"] = pagerank_arr
thermal_checkpoint(label="post degree + PageRank")
_print_resource("post degree + PageRank")

print(f"\nReal out-degree:  mean={node_df['out_degree'].mean():.2f}, max={node_df['out_degree'].max():,}")
print(f"Real in-degree :  mean={node_df['in_degree'].mean():.2f}, max={node_df['in_degree'].max():,}")
print(f"Real PageRank  :  mean={node_df['pagerank'].mean():.6e}, max={node_df['pagerank'].max():.6e}")

# ============================================================================
# 4. Real Is-Laundering exposure label per node.
# ============================================================================
laund = trans[trans["Is Laundering"] == 1]
exposed_from = laund["From Bank"].astype(str) + "|" + laund["Account"].astype(str)
exposed_to = laund["To Bank"].astype(str) + "|" + laund["Account.1"].astype(str)
exposed_keys = set(exposed_from) | set(exposed_to)
node_df["is_laundering_exposed"] = node_df["node_key"].isin(exposed_keys).astype("int8")
base_rate_all = float(node_df["is_laundering_exposed"].mean())
print(f"\nReal base rate, ALL {n_nodes:,} nodes: {100 * base_rate_all:.4f}% Is-Laundering-exposed "
      f"({int(node_df['is_laundering_exposed'].sum()):,} real exposed accounts).")

del trans, laund, exposed_from, exposed_to
gc.collect()
_print_resource("post label assignment")

# ============================================================================
# 5. Real train/test split of accounts -- identical method to Notebook 2.
# ============================================================================
from sklearn.model_selection import train_test_split

exposure = node_df["is_laundering_exposed"].to_numpy()
class_counts = pd.Series(exposure).value_counts()
print(f"\nReal node-level exposure class counts: {class_counts.to_dict()}")
strat = exposure if class_counts.min() >= 2 else None
if strat is None:
    print("WARNING: fewer than 2 real nodes in one exposure class -- stratified split disabled for this run.")

train_idx, test_idx = train_test_split(
    np.arange(n_nodes), test_size=0.25, random_state=RANDOM_SEED, stratify=strat
)
train_mask = np.zeros(n_nodes, dtype=bool)
train_mask[train_idx] = True
node_df["split"] = np.where(train_mask, "train", "test")
train_df = node_df[node_df["split"] == "train"]
test_df = node_df[node_df["split"] == "test"].copy()
base_rate_train = float(train_df["is_laundering_exposed"].mean())
base_rate_test = float(test_df["is_laundering_exposed"].mean())
print(f"Real split: {len(train_df):,} train nodes ({100*base_rate_train:.4f}% exposed), "
      f"{len(test_df):,} test nodes ({100*base_rate_test:.4f}% exposed).")

# ============================================================================
# 6. Real per-signal network-lift ratio -- identical method to Notebook 2 -- plus the memory-efficient
#    vectorized 2-hop proximity signal (header Section item 3).
# ============================================================================
def lift_ratio_report(signal_name, train_series, test_series, test_exposed, test_base_rate, pctl=0.90):
    threshold = float(train_series.quantile(pctl))
    flagged = test_series >= threshold
    n_flagged = int(flagged.sum())
    if n_flagged == 0:
        print(f"  {signal_name}: 0 real TEST nodes at/above the real train-fit {pctl:.0%} threshold "
              f"({threshold:.6g}) -- lift ratio UNDEFINED.")
        return {"signal": signal_name, "threshold": threshold, "n_test_flagged": 0,
                "n_test_flagged_exposed": 0, "flagged_exposure_rate": None, "lift_ratio": None}
    n_flagged_exposed = int(test_exposed[flagged.to_numpy()].sum())
    flagged_exposure_rate = float(test_exposed[flagged.to_numpy()].mean())
    lift = (flagged_exposure_rate / test_base_rate) if test_base_rate > 0 else None
    lift_str = f"{lift:.2f}x" if lift is not None else "UNDEFINED (real test base rate is 0)"
    print(f"  {signal_name}: real train-fit {pctl:.0%} threshold = {threshold:.6g}, {n_flagged:,} real TEST "
          f"nodes flagged, {100*flagged_exposure_rate:.4f}% real exposure among flagged -> lift = {lift_str}")
    return {"signal": signal_name, "threshold": threshold, "n_test_flagged": n_flagged,
            "n_test_flagged_exposed": n_flagged_exposed,
            "flagged_exposure_rate": flagged_exposure_rate, "lift_ratio": lift}

print(f"\nReal per-signal network-lift ratio at {DATASET_VARIANT} scale (train-fit threshold, test-measured lift):")
test_exposed_arr = test_df["is_laundering_exposed"].to_numpy()
stage_results = []
for col, label in [("out_degree", "Out-degree (top decile)"), ("in_degree", "In-degree (top decile)"),
                    ("total_degree", "Total-degree (top decile)"), ("pagerank", "PageRank (top decile)")]:
    stage_results.append(
        lift_ratio_report(label, train_df[col], test_df[col], test_exposed_arr, base_rate_test)
    )

def vectorized_frontier_bfs(edge_df, seed_ids, max_hops):
    """Real, memory-efficient multi-source bounded-BFS: two DIRECTED merges per hop (frontier-as-src ->
    find real dst; frontier-as-dst -> find real src) against the real edge table -- never a full
    undirected/deduplicated edge table (the real OOM this notebook's header discloses and fixes)."""
    visited = set(int(s) for s in seed_ids)
    frontier = pd.Series(list(visited), name="id", dtype="int64")
    for _hop in range(max_hops):
        out_next = edge_df.merge(frontier.to_frame("src"), on="src")["dst"]
        in_next = edge_df.merge(frontier.to_frame("dst"), on="dst")["src"]
        nxt = pd.concat([out_next, in_next], ignore_index=True).unique()
        new_ids = set(int(x) for x in nxt) - visited
        if not new_ids:
            break
        visited |= new_ids
        frontier = pd.Series(list(new_ids), name="id", dtype="int64")
    return visited

train_flagged_ids = train_df.loc[train_df["is_laundering_exposed"] == 1, "node_id"].to_numpy()
print(f"\nReal TRAIN-flagged seed accounts for the {EGO_HOPS}-hop proximity signal: {len(train_flagged_ids):,}")

assert_ram_safe(min_available_gb=4.0, label=f"before vectorized {EGO_HOPS}-hop frontier BFS")
with timer(f"real vectorized {EGO_HOPS}-hop frontier BFS from TRAIN-flagged accounts"):
    near_flagged_ids = vectorized_frontier_bfs(edge_df, train_flagged_ids, EGO_HOPS)
thermal_checkpoint(label="post 2-hop proximity BFS")

test_df["near_train_flagged"] = test_df["node_id"].isin(near_flagged_ids)
n_test_near = int(test_df["near_train_flagged"].sum())
if n_test_near == 0:
    print(f"  {EGO_HOPS}-hop proximity to a TRAIN-flagged account: 0 real TEST nodes within {EGO_HOPS} hops "
          f"-- lift ratio UNDEFINED.")
    stage_results.append({"signal": f"{EGO_HOPS}-hop proximity to a TRAIN-flagged account", "threshold": None,
                           "n_test_flagged": 0, "n_test_flagged_exposed": 0,
                           "flagged_exposure_rate": None, "lift_ratio": None})
else:
    near_exposure_rate = float(test_df.loc[test_df["near_train_flagged"], "is_laundering_exposed"].mean())
    lift = (near_exposure_rate / base_rate_test) if base_rate_test > 0 else None
    lift_str = f"{lift:.2f}x" if lift is not None else "UNDEFINED (real test base rate is 0)"
    print(f"  {EGO_HOPS}-hop proximity to a TRAIN-flagged account: {n_test_near:,} real TEST nodes within "
          f"{EGO_HOPS} hops, {100*near_exposure_rate:.4f}% real exposure among them -> lift = {lift_str}")
    stage_results.append({
        "signal": f"{EGO_HOPS}-hop proximity to a TRAIN-flagged account", "threshold": None,
        "n_test_flagged": n_test_near,
        "n_test_flagged_exposed": int(test_df.loc[test_df["near_train_flagged"], "is_laundering_exposed"].sum()),
        "flagged_exposure_rate": near_exposure_rate, "lift_ratio": lift,
    })
_print_resource("post 2-hop proximity signal")

ranked = sorted([r for r in stage_results if r["lift_ratio"] is not None],
                 key=lambda r: r["lift_ratio"], reverse=True)
print(f"\nReal ranking at {DATASET_VARIANT} scale (network-lift ratio):")
for r in ranked:
    print(f"  {r['signal']}: {r['lift_ratio']:.2f}x")
if not ranked:
    raise RuntimeError(
        "Every real candidate signal produced an UNDEFINED lift ratio on this run -- cannot select a "
        "real champion. Investigate the real class counts printed above before re-running."
    )
champion = ranked[0]
champion_name = champion["signal"]
print(f"\nReal champion (this run, {DATASET_VARIANT}): {champion_name} (lift = {champion['lift_ratio']:.2f}x)")
if nb2_champion_prior is not None:
    print(f"Real HI-Small (Notebook 2) prior champion was: {nb2_champion_prior!r} -- "
          f"{'CONFIRMED, same signal wins at both scales.' if nb2_champion_prior == champion_name else 'DIFFERENT at this real scale -- the real LI-Medium result above is authoritative, never silently overridden by the smaller-scale prior.'}")

# ============================================================================
# 7. Real bootstrap 95% CI on the champion's network-lift ratio -- closed-form multinomial resampling
#    (Section 9.3 technique #5/#6), drawn directly from the real observed 2x2 contingency table.
# ============================================================================
champ_result = next(r for r in stage_results if r["signal"] == champion_name)
n_test_total = int(len(test_df))
n_flag_exp = champ_result["n_test_flagged_exposed"]
n_flag_notexp = champ_result["n_test_flagged"] - n_flag_exp
n_notflag_exp = int(test_df["is_laundering_exposed"].sum()) - n_flag_exp
n_notflag_notexp = n_test_total - n_flag_exp - n_flag_notexp - n_notflag_exp
cell_counts = np.array([n_flag_exp, n_flag_notexp, n_notflag_exp, n_notflag_notexp], dtype="float64")
cell_probs = cell_counts / cell_counts.sum()
print(f"\nReal observed 2x2 contingency table (champion signal, {DATASET_VARIANT} TEST nodes): "
      f"flagged&exposed={n_flag_exp:,}, flagged&not={n_flag_notexp:,}, "
      f"not-flagged&exposed={n_notflag_exp:,}, not-flagged&not={n_notflag_notexp:,}")

N_BOOTSTRAP = 2000
assert_ram_safe(min_available_gb=2.0, label="before bootstrap multinomial resampling")
with timer(f"real bootstrap 95% CI, {N_BOOTSTRAP} multinomial resamples (closed-form, vectorized, Section 9.3 #5)"):
    rng = np.random.default_rng(RANDOM_SEED)
    resamples = rng.multinomial(n_test_total, cell_probs, size=N_BOOTSTRAP)  # (N_BOOTSTRAP, 4), one batched call
    r_flag_exp, r_flag_notexp, r_notflag_exp, r_notflag_notexp = resamples.T
    r_n_flagged = r_flag_exp + r_flag_notexp
    r_flagged_rate = np.divide(r_flag_exp, r_n_flagged, out=np.full(N_BOOTSTRAP, np.nan), where=r_n_flagged > 0)
    r_base_rate = (r_flag_exp + r_notflag_exp) / n_test_total
    r_lift = np.divide(r_flagged_rate, r_base_rate, out=np.full(N_BOOTSTRAP, np.nan), where=r_base_rate > 0)
    r_lift_valid = r_lift[~np.isnan(r_lift)]
    n_invalid = N_BOOTSTRAP - len(r_lift_valid)
    ci_low = float(np.percentile(r_lift_valid, 2.5)) if len(r_lift_valid) else None
    ci_high = float(np.percentile(r_lift_valid, 97.5)) if len(r_lift_valid) else None
    bootstrap_mean = float(np.mean(r_lift_valid)) if len(r_lift_valid) else None
print(f"Real bootstrap 95% CI on champion lift ratio: [{ci_low:.2f}x, {ci_high:.2f}x] "
      f"(bootstrap mean {bootstrap_mean:.2f}x, point estimate {champion['lift_ratio']:.2f}x, "
      f"{n_invalid}/{N_BOOTSTRAP} resamples had an undefined lift and were excluded).")
thermal_checkpoint(label="post bootstrap CI")

# ============================================================================
# 8. Real illustrative bounded ego-networks (identical method to Notebook 2, re-run here at real scale --
#    networkx used ONLY for these small, capped subgraphs, never the full graph).
# ============================================================================
example_seeds = (
    train_df[train_df["is_laundering_exposed"] == 1]
    .sort_values("total_degree", ascending=False)
    .head(3)["node_id"].tolist()
)
key_by_id = node_df.set_index("node_id")["node_key"]
example_ego_networks = []
print(f"\nReal illustrative bounded ego-networks ({EGO_HOPS}-hop, capped at {EGO_NODE_CAP:,} nodes):")
for seed in example_seeds:
    assert_ram_safe(min_available_gb=2.0, label=f"before bounded ego-network for seed node {seed}")
    local_ids = vectorized_frontier_bfs(edge_df, [seed], EGO_HOPS)
    local_edges = edge_df[edge_df["src"].isin(local_ids) & edge_df["dst"].isin(local_ids)]
    n_real_nodes = len(local_ids)
    truncated = n_real_nodes > EGO_NODE_CAP
    if truncated:
        Gego = nx.Graph()
        Gego.add_edges_from(zip(local_edges["src"], local_edges["dst"]))
        Gego.add_node(seed)
        dist = nx.single_source_shortest_path_length(Gego, seed)
        keep = set(sorted(dist, key=dist.get)[:EGO_NODE_CAP])
        local_edges = local_edges[local_edges["src"].isin(keep) & local_edges["dst"].isin(keep)]
        n_kept = len(keep)
    else:
        n_kept = n_real_nodes
    example_ego_networks.append({
        "seed_node_key": key_by_id.loc[seed], "real_nodes_before_cap": n_real_nodes,
        "nodes_after_cap": n_kept, "edges_after_cap": int(len(local_edges)), "truncated": bool(truncated),
    })
    print(f"  Seed {key_by_id.loc[seed]}: {n_real_nodes:,} real nodes within {EGO_HOPS} hops "
          f"({'TRUNCATED to ' + format(n_kept, ',') if truncated else 'under the cap, kept in full'}), "
          f"{len(local_edges):,} real edges in the capped subgraph.")
_print_resource("post illustrative ego-network construction")

# ============================================================================
# TWO-GATE VERDICT -- criteria fixed here, before being presented; never adjusted after seeing results.
# ============================================================================
print("\n" + "=" * 78)
print("TWO-GATE VERDICT")
print("=" * 78)

gate1_checks = {
    "all_candidate_signals_computed": len(stage_results) == 5,
    "pagerank_implementation_self_test_passed": _max_abs_diff <= 1e-4,
    "train_test_sizes_sum_to_total": (len(train_df) + len(test_df)) == n_nodes,
    "champion_selected_from_defined_lift_ratios_only": champion["lift_ratio"] is not None,
    "ram_safety_gates_passed_every_heavy_step": True,  # structurally true -- assert_ram_safe() raises and halts execution otherwise, so reaching this line means every gate above already passed for real
    "leakage_discipline_train_fit_test_measured": True,  # structural -- every threshold/seed-set above was built from train_df only (Section 6), by construction
}
gate1_pass = all(gate1_checks.values())
print("Gate 1 (structural [CHECK] -- did the real pipeline run correctly, real mechanics):")
for check, result in gate1_checks.items():
    print(f"  [{'PASS' if result else 'FAIL'}] {check}")
print(f"  Gate 1 verdict: {'PASS' if gate1_pass else 'FAIL'}")

gate2_checks = {
    "bootstrap_ci95_lower_bound_excludes_no_lift": (ci_low is not None) and (ci_low > 1.0),
    "bootstrap_had_at_least_95pct_valid_resamples": (N_BOOTSTRAP - n_invalid) / N_BOOTSTRAP >= 0.95,
}
gate2_pass = all(gate2_checks.values())
print("Gate 2 (statistical-robustness -- fixed criteria: the champion's real bootstrap 95% CI lower bound "
      "must exceed 1.0x (no real lift over the base rate), on a bootstrap with at least 95% valid "
      "resamples, a disclosed pre-registered bar, not tuned after seeing this run's numbers):")
for check, result in gate2_checks.items():
    print(f"  [{'PASS' if result else 'FAIL'}] {check}")
print(f"  Gate 2 verdict: {'PASS' if gate2_pass else 'FAIL'}")

overall_verdict = "PASS" if (gate1_pass and gate2_pass) else "FAIL"
print(f"\nOVERALL TWO-GATE VERDICT ({DATASET_VARIANT}): {overall_verdict}")

# ============================================================================
# Deployable Scoring Service -- real FastAPI app, self-tested via TestClient (in-process), bit-identical
# check against direct computation of the SAME champion flagging rule.
# ============================================================================
fastapi_self_test_report = None
champion_is_threshold_signal = champion_name in (
    "Out-degree (top decile)", "In-degree (top decile)", "Total-degree (top decile)", "PageRank (top decile)"
)
try:
    from fastapi import FastAPI
    from fastapi.testclient import TestClient
    from pydantic import BaseModel

    champ_col = {"Out-degree (top decile)": "out_degree", "In-degree (top decile)": "in_degree",
                 "Total-degree (top decile)": "total_degree", "PageRank (top decile)": "pagerank"}.get(champion_name)
    champ_threshold = champion["threshold"]

    class AccountFeatures(BaseModel):
        out_degree: float = 0.0
        in_degree: float = 0.0
        total_degree: float = 0.0
        pagerank: float = 0.0
        near_train_flagged_account: bool = False

    def score_account(record: dict) -> dict:
        """The SAME real flagging rule used for this notebook's own batch evaluation above -- this
        identity is what makes the self-test meaningful (Lesson #4 discipline, unchanged for a
        structural-signal rule rather than a trained classifier)."""
        if champion_is_threshold_signal:
            flagged = bool(record[champ_col] >= champ_threshold)
        else:
            flagged = bool(record["near_train_flagged_account"])
        return {"flagged": flagged, "champion_signal": champion_name,
                "champion_lift_ratio": champion["lift_ratio"]}

    app = FastAPI(title=f"BP3 Network-Signal Scoring Service ({DATASET_VARIANT})")

    @app.post("/score")
    def score_endpoint(acct: AccountFeatures):
        return score_account(acct.model_dump())

    with timer("FastAPI self-test (TestClient, real TEST rows, bit-identical check)"):
        client = TestClient(app)
        SELF_TEST_SAMPLE_SIZE = min(2000, len(test_df))
        self_test_df = test_df.sample(n=SELF_TEST_SAMPLE_SIZE, random_state=RANDOM_SEED)

        n_mismatch = 0
        for _, row in self_test_df.iterrows():
            record = {"out_degree": float(row["out_degree"]), "in_degree": float(row["in_degree"]),
                       "total_degree": float(row["total_degree"]), "pagerank": float(row["pagerank"]),
                       "near_train_flagged_account": bool(row["near_train_flagged"])}
            direct = score_account(record)
            resp = client.post("/score", json=record)
            api_result = resp.json()
            if api_result["flagged"] != direct["flagged"]:
                n_mismatch += 1

    fastapi_self_test_report = {"rows_checked": int(SELF_TEST_SAMPLE_SIZE), "mismatches": int(n_mismatch)}
    print(f"Real FastAPI self-test: {SELF_TEST_SAMPLE_SIZE} rows checked, {n_mismatch} mismatches.")
    print("SELF-TEST PASS (bit-identical flag decisions)" if n_mismatch == 0 else
          "SELF-TEST FAIL -- API path diverges from direct scoring, investigate before deploying")
except ImportError as e:
    print(f"SKIPPED FastAPI deployable service -- not importable on this machine ({e}). "
          f"ACTION NEEDED: pip install fastapi httpx.")

# ============================================================================
# Save Real Artifacts
# ============================================================================
_print_resource("before saving artifacts")
variant_tag = DATASET_VARIANT.lower().replace("-", "_")

rule_artifact = {
    "champion_signal": champion_name,
    "champion_column": champ_col if champion_is_threshold_signal else "near_train_flagged",
    "is_threshold_signal": champion_is_threshold_signal,
    "threshold": champion["threshold"],
    "note": "BP3 has no trained ML model (Notebook 2/3 header) -- this is a RULE artifact (a threshold or "
            "a set-membership test), not a pickled classifier. NB4's reporting layer and any deployment "
            "reads this file directly, never a .pkl model object.",
}
rule_path = MODELS_DIR / f"bp3_notebook3_champion_rule_{variant_tag}.json"
with open(rule_path, "w", encoding="utf-8") as f:
    json.dump(rule_artifact, f, indent=2)

# ----------------------------------------------------------------------------------------
# HARDENING ADDITION (2026-10-02): persist the real, already-computed per-account
# `near_train_flagged` boolean (Section 6's own `near_flagged_ids` set, produced by the real
# vectorized 2-hop frontier BFS above -- no new logic, no new computation) for EVERY real
# node in the graph, not just the real TEST-split subset Section 6 used for lift-ratio
# scoring. This closes the real gap this BP's own Dockerfile already disclosed ("ACTION
# NEEDED ... the real TRAIN-flagged seed-account set [proximity membership] needs to be
# persisted alongside the rule JSON for a deployed service to compute real proximity at
# inference time -- Notebook 3 computes this set in-kernel but does not currently persist
# it to disk"). Saved as Parquet, keyed by the real `node_key` (Bank|Account composite
# string, identical format to every other real key in this BP), per this platform's own
# WARP Parquet-over-CSV standard (master-execution-plan Section 9.2) -- at ~2.03M real rows
# this is a real, bounded, reused-not-recreated artifact, not a one-off CSV.
# ----------------------------------------------------------------------------------------
with timer("persist real per-account near_train_flagged lookup (hardening addition)"):
    near_train_flagged_lookup = pd.DataFrame({
        "node_key": node_df["node_key"].to_numpy(),
        "near_train_flagged": node_df["node_id"].isin(near_flagged_ids).to_numpy(),
    })
    lookup_path = MODELS_DIR / f"bp3_notebook3_near_train_flagged_lookup_{variant_tag}.parquet"
    near_train_flagged_lookup.to_parquet(lookup_path, index=False)
print(f"Real near_train_flagged lookup ({len(near_train_flagged_lookup):,} real accounts) saved "
      f"to: {lookup_path}")


with timer("save real Notebook 3 validation report"):
    report = {
        "bp_id": "BP3",
        "bp_name": "Transaction Network & Graph Intelligence",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_variant": DATASET_VARIANT,
        "random_seed": RANDOM_SEED,
        "primary_metric": PRIMARY_METRIC,
        "n_nodes": int(n_nodes),
        "n_distinct_edges": int(n_distinct_edges),
        "n_transaction_edges": int(n_edges_raw),
        "base_rate_all": base_rate_all,
        "n_train_nodes": int(len(train_df)),
        "n_test_nodes": int(len(test_df)),
        "base_rate_train": base_rate_train,
        "base_rate_test": base_rate_test,
        "pagerank_self_test_max_abs_diff_vs_networkx": _max_abs_diff,
        "hi_small_stage_a_prior_champion": nb2_champion_prior,
        "stage_results": stage_results,
        "champion_name": champion_name,
        "test_metrics": {"lift_ratio": champion["lift_ratio"]},
        "bootstrap": {"n_resamples": N_BOOTSTRAP, "n_invalid_resamples": int(n_invalid),
                      "ci95_low": ci_low, "ci95_high": ci_high, "bootstrap_mean": bootstrap_mean},
        "example_ego_networks": example_ego_networks,
        "gate1_structural_checks": gate1_checks,
        "gate1_verdict": "PASS" if gate1_pass else "FAIL",
        "gate2_statistical_robustness_checks": gate2_checks,
        "gate2_verdict": "PASS" if gate2_pass else "FAIL",
        "overall_verdict": overall_verdict,
        "fastapi_self_test": fastapi_self_test_report,
        "no_trained_model_note": "BP3 has no trained ML model -- every candidate is a directly-"
                                  "interpretable structural signal; SHAP/LIME are N/A, not attempted.",
        "scaling_engineering_note": "networkx avoided for the full graph (Lesson #21/#26) -- hand-rolled "
                                     "vectorized sparse PageRank (self-tested above) and a memory-efficient "
                                     "frontier-expansion BFS used instead; networkx used only for the small "
                                     "capped illustrative ego-networks (Section 8).",
    }
    report_path = REPORTS_DIR / f"bp3_notebook3_validation_report_{variant_tag}.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

print(f"Real champion rule saved to: {rule_path}")
print(f"Real validation report saved to: {report_path}")

# ============================================================================
# Notebook 3 Summary
# ============================================================================
print("\n" + "=" * 78)
print(f"BP3 -- Notebook 3 Summary ({DATASET_VARIANT}, real, this run)")
print("=" * 78)
print(f"Nodes: {n_nodes:,}  Distinct edges: {n_distinct_edges:,}  Base rate: {100*base_rate_all:.4f}%")
print(f"Champion              : {champion_name}")
print(f"Real lift ratio        : {champion['lift_ratio']:.2f}x  (bootstrap 95% CI [{ci_low:.2f}x, {ci_high:.2f}x])")
print(f"Two-gate verdict       : Gate1={'PASS' if gate1_pass else 'FAIL'}  Gate2={'PASS' if gate2_pass else 'FAIL'}  "
      f"Overall={overall_verdict}")
print("=" * 78)
print("NEXT: notebooks/bp3_network_graph_intelligence/04_compliance_impact_reporting_packaging_SINGLE_CELL.py "
      "-- reads this notebook's real validation report + Notebook 2's real Stage A summary, and produces "
      "BP3's Word/Excel/HTML/PPTX compliance package (requires a new report_builder.py writer path for "
      "this BP's structural-lift-ratio output shape, scoped separately -- see that notebook's own header).")
print("\n### STAGE MARKER 4/4 COMPLETE -- Notebook 3 fully finished, all artifacts saved ###")
